In [ ]:
import pandas as pd
import numpy as np


In [ ]:
master_file = "series_part_mapping.xlsx"
prod_file   = "production data.xlsx"
tool_file   = "Tool list of Mould fixation-VT.xlsx"

df_master    = pd.read_excel(master_file, sheet_name="Master Sheet")
df_inventory = pd.read_excel(master_file, sheet_name="Inventory")
df_sheet10   = pd.read_excel(master_file, sheet_name="Sheet10")
df_fg        = pd.read_excel(master_file, sheet_name="Sheet4")

df_prod = pd.read_excel(prod_file)
df_tool = pd.read_excel(tool_file)


In [ ]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(".", "", regex=False)
    )
    return df

def norm(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


In [ ]:
df_master    = normalize_columns(df_master)
df_inventory = normalize_columns(df_inventory)
df_sheet10   = normalize_columns(df_sheet10)
df_fg        = normalize_columns(df_fg)
df_prod      = normalize_columns(df_prod)
df_tool      = normalize_columns(df_tool)


In [ ]:
def get_fg_stock_by_series(series):
    s = norm(series)

    r = df_inventory[df_inventory["material"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["unrestricted"]):
        return r.iloc[0]["unrestricted"]

    r = df_sheet10[df_sheet10["material"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["unrestricted"]):
        return r.iloc[0]["unrestricted"]

    r = df_fg[df_fg["series"].astype(str).apply(norm) == s]
    if not r.empty and not pd.isna(r.iloc[0]["fg_quantity"]):
        return r.iloc[0]["fg_quantity"]

    return 0


In [ ]:
def get_machine_by_series(series):
    s = norm(series)
    r = df_tool[df_tool["part_no"].astype(str).apply(norm) == s]
    if not r.empty:
        return r.iloc[0]["machine_no"]
    return None


In [ ]:
def get_time_required_by_series(series, qty):
    s = norm(series)
    r = df_prod[df_prod["part_no"].astype(str).apply(norm) == s]

    if r.empty:
        return None

    pph = r.iloc[0]["part_per_hour"]
    if pd.isna(pph) or pph <= 0:
        return None

    return qty / pph   # hours


In [ ]:
MAX_MINUTES = 22 * 60
machine_load = {}
PLAN = []


In [ ]:
for i, row in df_master.iterrows():

    if i % 5000 == 0:
        print(f"Processed {i} rows...")

    series = row["series"]

    monthly_req = row["monthely_requirement-jan"] if not pd.isna(row["monthely_requirement-jan"]) else 0
    min_req     = row["minimum_requirement"] if not pd.isna(row["minimum_requirement"]) else 0
    dispatch    = row["dispatch"] if not pd.isna(row["dispatch"]) else 0

    fg_stock = get_fg_stock_by_series(series)
    available_fg = fg_stock - dispatch

    net_req = monthly_req + min_req - available_fg
    if net_req <= 0:
        continue

    daily_demand = monthly_req / 28 if monthly_req > 0 else 0

    # ---- PRIORITY LOGIC (UNCHANGED CORE) ----
    if available_fg < daily_demand:
        planned_qty = daily_demand
    elif daily_demand <= 50:
        planned_qty = min(5 * daily_demand, net_req)
    else:
        planned_qty = net_req

    if planned_qty <= 0:
        continue

    machine = get_machine_by_series(series)
    if machine is None:
        continue

    time_hrs = get_time_required_by_series(series, planned_qty)
    if time_hrs is None:
        continue

    time_mins = time_hrs * 60
    used_mins = machine_load.get(machine, 0)

    # ---- MACHINE CAPACITY CHECK ----
    if used_mins + time_mins > MAX_MINUTES:
        continue   # skip if machine overloaded

    machine_load[machine] = used_mins + time_mins

    PLAN.append({
        "Machine": machine,
        "Series": series,
        "Qty": int(round(planned_qty)),
        "Time Required (hrs)": round(time_hrs, 2)
    })


In [ ]:
df_plan = pd.DataFrame(PLAN)
df_plan = df_plan.sort_values("Machine").reset_index(drop=True)
df_plan


In [ ]:
machine_util = (
    pd.DataFrame.from_dict(machine_load, orient="index", columns=["Used Minutes"])
    .reset_index()
    .rename(columns={"index": "Machine"})
)

machine_util["Used Hours"] = machine_util["Used Minutes"] / 60
machine_util
